# NB04 — Transfer Learning, Fine-Tuning y evaluación de CNN
**Correspondencia: Semanas 12–13**


## 1. Preparación del entorno


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

print("TensorFlow:", tf.__version__)
print("GPU disponible:", tf.config.list_physical_devices("GPU"))


## 2. Conectar Google Drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 3. Definir el dataset

Utilice el dataset preparado en NB02. Para esta etapa se recomienda disponer de conjuntos independientes de **train, validation y test**.

Estructura esperada:

```text
dataset_final/
├── train/
├── validation/
└── test/
```

Cada subconjunto debe contener las mismas carpetas de clases.


In [ ]:
BASE_DIR = "/content/drive/MyDrive/MachineLearning2026/dataset_final"

TRAIN_DIR = f"{BASE_DIR}/train"
VAL_DIR   = f"{BASE_DIR}/validation"
TEST_DIR  = f"{BASE_DIR}/test"

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
SEED = 42


## 4. Cargar train, validation y test


In [ ]:
train_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    seed=SEED
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    VAL_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    TEST_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False
)

class_names = train_ds.class_names
NUM_CLASSES = len(class_names)

print("Clases:", class_names)
print("Número de clases:", NUM_CLASSES)


## 5. Optimizar el pipeline de datos


In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)
test_ds = test_ds.prefetch(AUTOTUNE)


## 6. Data Augmentation

Las transformaciones se aplicarán únicamente durante entrenamiento.


In [ ]:
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.10),
    layers.RandomZoom(0.10)
], name="data_augmentation")


## 7. Cargar un modelo preentrenado

Utilizaremos **MobileNetV2** como extractor de características. Los pesos provienen de ImageNet.


In [ ]:
base_model = keras.applications.MobileNetV2(
    input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3),
    include_top=False,
    weights="imagenet"
)

base_model.trainable = False

print("Capas del modelo base:", len(base_model.layers))


## 8. Construir el modelo mediante Transfer Learning


In [ ]:
inputs = keras.Input(shape=(IMG_SIZE[0], IMG_SIZE[1], 3))

x = data_augmentation(inputs)
x = keras.applications.mobilenet_v2.preprocess_input(x)

x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.30)(x)

outputs = layers.Dense(
    NUM_CLASSES,
    activation="softmax"
)(x)

model = keras.Model(inputs, outputs)

model.summary()


## 9. Compilar el modelo


In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)


## 10. Entrenamiento con la base congelada


In [ ]:
EPOCHS_TRANSFER = 10

history_transfer = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_TRANSFER
)


## 11. Evaluar la primera etapa


In [ ]:
val_loss_transfer, val_acc_transfer = model.evaluate(
    val_ds,
    verbose=0
)

print(f"Validation loss: {val_loss_transfer:.4f}")
print(f"Validation accuracy: {val_acc_transfer:.4f}")


## 12. Visualizar curvas de aprendizaje


In [ ]:
hist_transfer = pd.DataFrame(history_transfer.history)

plt.figure(figsize=(8,5))
plt.plot(hist_transfer["accuracy"], label="Entrenamiento")
plt.plot(hist_transfer["val_accuracy"], label="Validación")
plt.xlabel("Época")
plt.ylabel("Accuracy")
plt.title("Transfer Learning — Accuracy")
plt.legend()
plt.show()

plt.figure(figsize=(8,5))
plt.plot(hist_transfer["loss"], label="Entrenamiento")
plt.plot(hist_transfer["val_loss"], label="Validación")
plt.xlabel("Época")
plt.ylabel("Loss")
plt.title("Transfer Learning — Loss")
plt.legend()
plt.show()


## 13. Fine-Tuning

Ahora desbloquearemos una parte de MobileNetV2. Se utilizará un **learning rate menor** para realizar ajustes finos sin modificar bruscamente los pesos preentrenados.


In [ ]:
base_model.trainable = True

FINE_TUNE_AT = 100

for layer in base_model.layers[:FINE_TUNE_AT]:
    layer.trainable = False

print("Capas totales:", len(base_model.layers))
print("Capas entrenables:", sum(layer.trainable for layer in base_model.layers))


## 14. Recompilar para Fine-Tuning


In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)


## 15. Entrenamiento Fine-Tuning


In [ ]:
EPOCHS_FINE = 10

history_fine = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_FINE
)


## 16. Comparar Transfer Learning y Fine-Tuning


In [ ]:
val_loss_fine, val_acc_fine = model.evaluate(
    val_ds,
    verbose=0
)

comparacion = pd.DataFrame({
    "Etapa": ["Transfer Learning", "Fine-Tuning"],
    "Validation_accuracy": [
        val_acc_transfer,
        val_acc_fine
    ]
})

comparacion


## 17. Evaluación final con Test

El conjunto de **test** se utiliza al final, una vez seleccionada la configuración del modelo.


In [ ]:
test_loss, test_accuracy = model.evaluate(
    test_ds,
    verbose=0
)

print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_accuracy:.4f}")


## 18. Obtener predicciones sobre Test


In [ ]:
y_true = []
y_pred = []
y_conf = []

for images, labels in test_ds:
    probabilities = model.predict(images, verbose=0)
    predictions = np.argmax(probabilities, axis=1)
    confidence = np.max(probabilities, axis=1)

    y_true.extend(labels.numpy())
    y_pred.extend(predictions)
    y_conf.extend(confidence)

y_true = np.array(y_true)
y_pred = np.array(y_pred)
y_conf = np.array(y_conf)

print("Predicciones generadas:", len(y_pred))


## 19. Matriz de confusión


In [ ]:
cm = confusion_matrix(y_true, y_pred)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=class_names
)

fig, ax = plt.subplots(figsize=(8,8))
disp.plot(ax=ax, xticks_rotation=45)
plt.title("Matriz de confusión — Test")
plt.show()


## 20. Precision, Recall y F1-score


In [ ]:
print(
    classification_report(
        y_true,
        y_pred,
        target_names=class_names,
        digits=4
    )
)


## 21. Revisar errores del modelo


In [ ]:
errores = np.where(y_true != y_pred)[0]

print("Cantidad de errores:", len(errores))

df_errores = pd.DataFrame({
    "Clase_real": [class_names[i] for i in y_true[errores]],
    "Clase_predicha": [class_names[i] for i in y_pred[errores]],
    "Confianza": y_conf[errores]
})

df_errores.sort_values(
    "Confianza",
    ascending=False
).head(15)


## 22. Comparar con la CNN desarrollada desde cero

Complete los resultados obtenidos en NB03 para comparar ambas estrategias.


In [ ]:
# Reemplace los valores de ejemplo por los resultados reales de NB03.

comparacion_modelos = pd.DataFrame({
    "Modelo": [
        "CNN desde cero",
        "MobileNetV2 + Fine-Tuning"
    ],
    "Accuracy_test": [
        np.nan,       # completar con resultado NB03
        test_accuracy
    ],
    "Parametros": [
        np.nan,       # completar con resultado NB03
        model.count_params()
    ]
})

comparacion_modelos


## 23. Guardar el modelo definitivo

La versión seleccionada será utilizada posteriormente en las etapas de optimización, interoperabilidad e implementación.


In [ ]:
MODEL_PATH = "/content/drive/MyDrive/MachineLearning2026/modelo_cnn_definitivo.keras"

model.save(MODEL_PATH)

print("Modelo guardado en:")
print(MODEL_PATH)


## 24. Registrar el contrato básico del modelo


In [ ]:
contrato_modelo = {
    "modelo": "MobileNetV2 + Fine-Tuning",
    "input_shape": [224, 224, 3],
    "tipo_entrada": "RGB",
    "preprocesamiento": "MobileNetV2 preprocess_input",
    "numero_clases": NUM_CLASSES,
    "clases": class_names,
    "salida": "probabilidades Softmax",
    "test_accuracy": float(test_accuracy)
}

contrato_modelo


## 25. Actividad

A partir del proyecto integrador:

1. Entrene una arquitectura mediante Transfer Learning.
2. Registre el modelo preentrenado utilizado.
3. Explique qué capas permanecieron congeladas inicialmente.
4. Realice Fine-Tuning utilizando un learning rate menor.
5. Compare el desempeño antes y después del Fine-Tuning.
6. Evalúe el modelo definitivo utilizando el conjunto de test.
7. Analice la matriz de confusión.
8. Analice Precision, Recall y F1-score por clase.
9. Identifique las clases que presentan mayores dificultades.
10. Compare el resultado con la CNN desarrollada desde cero en NB03.
11. Justifique qué modelo seleccionará como modelo definitivo.
12. Guarde y versione el modelo seleccionado.


## 26. Base para la Evaluación 3

El resultado de este notebook debe permitir seleccionar y justificar el **modelo CNN definitivo** del proyecto integrador.

**CNN inicial → Transfer Learning → Fine-Tuning → evaluación con Test → análisis por clase → comparación → selección del modelo definitivo**

El modelo seleccionado será utilizado en NB05 para abordar **optimización, cuantificación, poda e interoperabilidad mediante ONNX**.
